# From linear classification to neural networks

**Lab 5 · Building and fitting a neural network**

In Labs 3 and 4, you implemented predictors and classifiers, fitted their parameters,
and examined their empirical losses. Here you will do the same for a neural network.
We supply the familiar affine classifier on a new dataset. **You will implement the
network, use automatic differentiation to obtain its gradients, adapt gradient
descent, and interpret the results using supplied plots.** Start with two hidden neurons, as
in the lecture's XOR example, and try a different initialisation of this same
network. Then widen the network to 100 hidden neurons.
Finally, build a network with **two hidden layers of 5 and 7 neurons**, as in the
lecture's deeper architecture. Compare these architectures and starting points
on the same observations.

These experiments illustrate two **heuristic strategies from the lecture**:
trying different initialisations and changing the architecture. We explore the
second strategy by increasing width, then by changing both width and depth
relative to the original two-neuron network. We will assess their effect using
the empirical losses, mistake counts and decision boundaries.

Use **Google Colab** and work from top to bottom; no GPU is needed.
Write your code in each **YOUR CODE** cell before opening **Show answer**.
The answer gives a solution, or refers you to a solution you already wrote.
Opening an answer does not execute it: enter the code in the exercise cell and run it.
All plotting code is supplied; run those cells and interpret the figures.

**To repeat an experiment from the beginning, rerun its parameter-initialisation
cell and then its fitting cell.** Rerunning only fitting continues from the current
parameters. The stated seeds make each initialisation repeatable without changing
the existing dataset.

Use the parameter, function and result names requested in the exercises so that
the supplied checks and plots can find them. Names of local intermediate variables
are up to you. We use the functions, loops, lists and tensor operations from Labs
1–4; new operations are explained before you need them.

## 1. Generate and visualise the data

We generate **100 observations**, arranged in four Gaussian clouds. Each cloud has 25 observations, with independent coordinate noise of standard deviation $0.225$.

| Cloud centre | Class | Observations |
|---|---:|---:|
| $(0,0)$ | 0 | 25 |
| $(1,1)$ | 0 | 25 |
| $(0,1)$ | 1 | 25 |
| $(1,0)$ | 1 | 25 |

The four centres are exactly the XOR inputs from the lectures. Diagonally opposite clouds share a label. This is a noisy version of the XOR arrangement from the lectures: the inputs are points in the Euclidean plane, and each label records the class assigned to the cloud that generated the observation. Blue circles show class 0 and orange triangles show class 1, as in Lab 4.

The tensor `X` contains the two input coordinates and has shape `(N, 2)`. The labels in `y` have shape `(N, 1)`. Run the supplied cell to create and visualise the data.

In [ ]:
import torch as t
import matplotlib.pyplot as plt
t.manual_seed(1)  # Fix the random seed for this notebook.

N = 100
# Four Gaussian clouds in an XOR arrangement: 25 observations per cloud.
# spread is the standard deviation of each coordinate, not a variance.
spread = 0.225
x_class_0 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([0.0, 0.0]),
    t.randn(N // 4, 2) * spread + t.tensor([ 1.0,  1.0])
], 0)
x_class_1 = t.cat([
    t.randn(N // 4, 2) * spread + t.tensor([0.0, 1.0]),
    t.randn(N // 4, 2) * spread + t.tensor([1.0, 0.0])
], 0)
X = t.cat([x_class_0, x_class_1], 0)              # Shape: (100, 2).
y = t.cat([t.zeros(N // 2, 1), t.ones(N // 2, 1)], 0)  # Shape: (100, 1).

# Show observed classes using the same colours and markers as in Lab 4.
plt.figure()
plt.scatter(x=x_class_0[:, 0], y=x_class_0[:, 1], color='tab:blue', label='Class 0')
plt.scatter(x=x_class_1[:, 0], y=x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
plt.xlabel('$x_1$')
plt.ylabel('$x_2$')
plt.axis('equal')
plt.legend()
plt.show()

**Supplied plotting tools.** Run the next cell once. Its code is
folded away because drawing the figures is not an exercise.
Later cells call these tools with your recorded losses, mistake
counts and boundary scores; you only need to run those cells and
interpret the pictures. `save_boundary(net)` records the current
network's scores on a drawing grid, so we can display that stage
after the parameters have changed. This grid adds no observations
to the dataset used for fitting.

In [ ]:
#@title Supplied plotting helpers — run this cell once; no changes are needed.
import math

# This grid is only for drawing pictures; it adds no fitting observations.
x1_limits = (X[:, 0].min().item() - 0.25, X[:, 0].max().item() + 0.25)
x2_limits = (X[:, 1].min().item() - 0.25, X[:, 1].max().item() + 0.25)
x1_grid, x2_grid = t.meshgrid(t.linspace(*x1_limits, 250),
                            t.linspace(*x2_limits, 250), indexing='xy')
X_grid = t.stack([x1_grid, x2_grid], dim=-1).reshape(-1, 2)

def save_boundary(network):
    return network(X_grid).reshape(x1_grid.shape).detach()

def save_affine_boundary(theta):
    return (theta[0] + X_grid @ theta[1:]).reshape(x1_grid.shape)

def plot_affine_curves(losses, mistakes):
    updates = range(len(losses))
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    axes[0].plot(updates, losses, color='0.35')
    axes[0].set(title='Affine model: empirical logistic loss', ylabel='Summed loss', ylim=(0, 75))
    axes[1].plot(updates, [value / math.log(2) for value in losses],
                 color='tab:purple', label=r'Loss / $\log 2$')
    axes[1].plot(updates, mistakes, color='tab:green', label='Mistakes')
    axes[1].set(title='Affine model: loss and mistakes', ylabel='Value', ylim=(0, 110))
    axes[1].legend()
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
    plt.show()

def plot_affine_boundaries(checkpoints):
    x1_line = t.tensor(x1_limits)
    fig, ax = plt.subplots(figsize=(8.5, 5.2), constrained_layout=True)
    ax.scatter(x_class_0[:, 0], x_class_0[:, 1], color='tab:blue', label='Class 0')
    ax.scatter(x_class_1[:, 0], x_class_1[:, 1], color='tab:orange', marker='^', label='Class 1')
    for (step, theta), style in zip(checkpoints.items(), (':', '--', '-.', '-')):
        b, w1, w2 = theta[:, 0]
        x2_line = -(b + w1 * x1_line) / w2
        ax.plot(x1_line, x2_line, color='black', linestyle=style,
                label=f'After {step} update' + ('s' if step != 1 else ''))
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$',
           title='Affine decision boundaries')
    ax.set_aspect('equal')
    ax.legend(loc='upper left', bbox_to_anchor=(1, 1))
    plt.show()

def plot_first_comparison(affine_losses, affine_mistakes, network_losses, network_mistakes):
    updates = range(len(network_losses))
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    axes[0].plot(updates, affine_losses, '--', color='0.4', label='Affine score')
    axes[0].plot(updates, network_losses, color='tab:purple', label='Two-neuron network')
    axes[0].set(title='Empirical logistic loss', ylabel='Summed loss')
    for ax, losses, mistakes, title in zip(
            axes[1:], [affine_losses, network_losses], [affine_mistakes, network_mistakes],
            ['Affine model: loss and mistakes', 'Two-neuron network: loss and mistakes']):
        ax.plot(updates, t.tensor(losses) / math.log(2), color='tab:purple', label=r'Loss / $\log 2$')
        ax.plot(updates, mistakes, color='tab:green', label='Mistakes')
        ax.set(title=title, ylabel='Value', ylim=(0, 1.1 * max(100, max(losses) / math.log(2))))
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
        ax.legend()
    plt.show()

def plot_network_curves(labels, loss_histories, error_histories):
    fig, axes = plt.subplots(1, 2, figsize=(12, 4), constrained_layout=True)
    styles = ['--', '-'] if len(labels) == 2 else [':', '--', '-']
    for label, losses, mistakes, style in zip(labels, loss_histories, error_histories, styles):
        updates = range(len(losses))
        axes[0].plot(updates, losses, style, color='tab:purple', label=label)
        axes[1].plot(updates, mistakes, style, color='tab:green', label=label)
    axes[0].set(title='Empirical logistic loss', ylabel='Summed loss')
    axes[1].set(title='Classification mistakes', ylabel='Mistakes')
    for ax in axes:
        ax.set_xlabel('Completed updates')
        ax.grid(alpha=0.2)
        ax.legend()
    plt.show()

def draw_boundary(ax, score_grid, title):
    classes = (score_grid > 0).to(t.float32)
    ax.contourf(x1_grid, x2_grid, classes,
                levels=[-0.5, 0.5, 1.5], colors=['#edf3fb', '#fff1e9'])
    ax.contour(x1_grid, x2_grid, score_grid, levels=[0], colors='black', linewidths=1)
    class_0 = y[:, 0] == 0
    class_1 = y[:, 0] == 1
    ax.scatter(X[class_0, 0], X[class_0, 1], color='tab:blue', s=18, label='Class 0')
    ax.scatter(X[class_1, 0], X[class_1, 1], color='tab:orange', marker='^', s=20, label='Class 1')
    ax.set(xlim=x1_limits, ylim=x2_limits, xlabel='$x_1$', ylabel='$x_2$', title=title)
    ax.set_aspect('equal')

def plot_boundaries(score_grids, labels, mistake_counts):
    width = {2: 10, 3: 13, 4: 15}[len(score_grids)]
    fig, axes = plt.subplots(1, len(score_grids), figsize=(width, 4), constrained_layout=True)
    for ax, scores, label, mistakes in zip(axes, score_grids, labels, mistake_counts):
        draw_boundary(ax, scores, f'{label}\nMistakes: {mistakes}')
    axes[0].legend(loc='upper left', fontsize=8)
    plt.show()

## 2. Reuse the linear classifier from Lab 4

Look at the four clouds. **Could a single straight decision boundary separate the four centres according to their labels?** Keep your prediction in mind while running the model.

The code below supplies the completed calculations from Lab 4: the stable empirical logistic loss, its gradient and the number of classification mistakes. You do not need to implement them again. Recall why we use `logsigmoid` to evaluate the logarithms directly.

As in Lab 4, we include the intercept using a column of ones. Here that design matrix is called `X_affine`, while `X` keeps the original coordinates for the later network. The functions are prefixed with `affine_` to keep the two calculations separate. Their mathematical definitions are unchanged.

The coefficient column contains $(b,w_1,w_2)$, and the scores are `X_affine @ theta`. The predicted label is 1 when the sigmoid of the score is greater than 0.5, and 0 otherwise, including at equality. This is equivalent to testing whether the score is strictly positive.

In [ ]:
# Supplied: the completed functions from Lab 4, with affine-specific names.
X_affine = t.cat([t.ones(N, 1), X], dim=1)  # Shape (N, 3), including the intercept.

def affine_loss(theta):
    scores = X_affine @ theta
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

def affine_grad(theta):
    scores = X_affine @ theta
    probabilities = t.sigmoid(scores)
    return X_affine.mT @ (probabilities - y)

def affine_mistakes(theta):
    probabilities = t.sigmoid(X_affine @ theta)
    predicted_labels = (probabilities > 0.5).to(y.dtype)
    return int((predicted_labels != y).sum().item())

### 2.1. Run gradient descent

Run the supplied update loop. We use Lab 4's zero initialisation and learning rate `0.01`, now allowing **1,000 updates**. Every update uses the full dataset.

As before, record the initial loss and mistake count, then record them again **after** each update. Entry `step` in each history therefore describes exactly `step` completed updates. We also store the parameter tensors at selected steps for the boundary plot.


**Recording results.** A Python list such as `[first_value]` starts with one entry;
`history.append(value)` adds another at the end. `history[0]` selects the first
entry, and `history[-1]` the last. Here entry `step` records that many completed
updates. `range(1, num_updates + 1)` includes 1 through `num_updates`, because the
upper endpoint of `range` is excluded. `.item()` extracts an ordinary Python
number from a tensor containing one value.

`affine_checkpoints = {}` creates an empty **dictionary**, which stores values
under keys. `affine_checkpoints[step] = affine_theta` stores the current parameter
tensor under the update number `step`.
The condition `step in (...)` checks whether `step` is one of the listed numbers.

The printed table uses **f-strings**: in `f'Update {step}'`, Python inserts the
value of `step` into the text. A format such as `{value:.3f}` displays three
decimal places; table-width and precision settings only affect the display.

In [ ]:
affine_theta = t.zeros(3, 1)
affine_learning_rate = 0.01
num_updates = 1000
affine_loss_history = [affine_loss(affine_theta).item()]
affine_error_history = [affine_mistakes(affine_theta)]
affine_checkpoints = {}

for step in range(1, num_updates + 1):
    affine_theta = affine_theta - affine_learning_rate * affine_grad(affine_theta)
    affine_loss_history.append(affine_loss(affine_theta).item())
    affine_error_history.append(affine_mistakes(affine_theta))

    # Supplied: save this tensor; each later update creates a new one.
    if step in (1, 10, 100, num_updates):
        affine_checkpoints[step] = affine_theta

print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(f'{step:7d}   {affine_loss_history[step]:23.6f}   {affine_error_history[step]:8d}')

### 2.2. Inspect the curves and the boundaries

Run the supplied plotting cells. These are the same quantities we examined in Lab 4:

- the summed empirical logistic loss;
- the loss divided by $\log 2$, alongside the number of mistakes;
- the decision boundaries at successive updates.

The division by $\log 2$ is only for display: **the updates use the original, unscaled loss**. Each mistake contributes at least $\log 2$ to that loss, so the scaled loss bounds the mistake count from above.

All affine boundaries have the form $b+w_1x_1+w_2x_2=0$. For these saved coefficients, $w_2$ is nonzero, so we can draw the lines using $x_2=-(b+w_1x_1)/w_2$, just as in Lab 4. This plotting code is supplied; focus on interpreting its output.

In [ ]:
# Supplied: the affine loss and mistake curves.
plot_affine_curves(affine_loss_history, affine_error_history)

In [ ]:
# Supplied: draw the saved affine boundaries and keep its final picture.
plot_affine_boundaries(affine_checkpoints)
affine_grid_scores = save_affine_boundary(affine_theta)

### Pause and interpret: what has gradient descent achieved?

1. Compare the initial and final empirical losses and mistake counts. Has the affine model fitted the observations well? The loss curve looks almost flat: does this mean that no updates have taken place? Use the printed values to read changes that are too small to see in the curves.
2. What happens to the boundary between updates 1, 100 and 1,000? Could more updates turn it into a boundary with several segments?
3. Why do the loss and the number of mistakes not necessarily decrease together?

Write down your answers before opening the explanation.

<details>
<summary>Show answer — interpreting the affine fit</summary>

**1. The loss decreases very little, but the mistake count first falls and then rises.** With the supplied seed, the empirical logistic loss changes from approximately $69.3147$ to $69.3083$. The mistake count starts at 50, falls to 33 after 3 updates (and remains at 33 after 4), then rises to 47 by update 100. The loss has already essentially stopped changing by update 100. The final model still misclassifies almost half of these observations.

The loss curve looks flat because the total decrease is only about $0.00645$. Initially, all coefficients are zero, so every score is zero and every predicted probability is $1/2$. The summed empirical loss is therefore $N\log 2\approx69.3147$. The purple curve in the right panel shows this loss divided by $\log 2$, so it starts at 100 and remains close to 100.

For this nearly symmetric XOR sample, the fitted affine scores remain close to zero: a single straight decision boundary cannot capture the main XOR pattern. Small sample asymmetries allow a slight reduction in the loss, but the probabilities remain near $1/2$. Small changes in these probabilities can move observations across the classification threshold and change their predicted labels. The empirical logistic loss is convex in the affine coefficients, so this behaviour is not due to a suboptimal local minimum. It illustrates a limitation of the affine model on these data.

**2. Updates change the coefficients of a straight line.** The boundary initially rotates and shifts; the lines at 100 and 1,000 updates almost coincide. Further updates can change its slope and intercept, but an affine score always has a straight boundary when its feature weights are nonzero. A single straight line cannot separate the four cloud centres according to their XOR labels. We need a richer family of scores to express a different shape.

**3. Gradient descent reduces the logistic loss, not directly the number of mistakes.** The logistic loss depends on the probabilities assigned to the observed labels. The mistake count only checks which side of the classification threshold each observation falls on. Probabilities can change without changing any labels, and a decrease in total loss can also accompany an increase in mistakes.

We see this directly here: between updates 4 and 100, the loss falls from approximately $69.3121$ to $69.3083$, while the mistake count rises from 33 to 47. Crossing the threshold changes an observation's contribution to the mistake count abruptly, but changes its logistic loss continuously. Gradient descent follows the derivatives of the total logistic loss; it does not choose each update to minimise the mistake count.

The final 47 mistakes are therefore the outcome of fitting the logistic loss in this run, not the smallest mistake count attainable by a straight boundary. Indeed, the earlier model already made only 33 mistakes, although its logistic loss was slightly larger.

</details>

## 3. Can we do better with a neural network?

Keep the **same observations, labels and empirical logistic loss**, and change the
model that produces the scores. In the lectures, the hidden weights and biases
made the feature functions adjustable. You will now implement that model and fit
all of its parameters.

In Lab 4, we wrote the gradient formula ourselves. Here, PyTorch's **automatic
differentiation** will calculate the derivatives for us. You will still implement
the gradient-descent updates. The next lecture explains **backpropagation**, the
algorithm behind this derivative calculation.

### Reuse the empirical logistic loss

Our Lab 4 loss function calculated the affine scores internally. The supplied
`loss_from_scores` instead takes already-computed scores and evaluates the same
empirical loss, using the labels in `y`:

$$\mathcal L_{\mathrm{log}} = \sum_{i=1}^N
\ell_{\mathrm{log}}(\sigma(s_i),y_i).$$

We can use it with either affine or neural-network scores. Recall Lab 4's numerical
stability issue: we again use `logsigmoid` to evaluate the logarithms directly.
**Pass scores to this function without first applying a sigmoid.** You do not need
to implement this familiar loss again.

In [ ]:
def loss_from_scores(scores):
    # scores contains classification scores, not probabilities or weights.
    # Evaluate sum_i ell_log(sigmoid(score_i), y_i), using the fixed labels y.
    # logsigmoid computes the logarithms directly, without forming probabilities.
    return -(y * t.nn.functional.logsigmoid(scores)
             + (1 - y) * t.nn.functional.logsigmoid(-scores)).sum()

### 3.1. From the architecture to parameter tensors

Start with **two input coordinates, two hidden neurons and one scalar output**,
matching the architecture of the lecture's XOR example. In the notes' notation,
$L=2$, $d_0=2$, $d_1=2$ and $d_2=1$. Here $N=100$ counts observations;
$d_1$ counts hidden neurons and will change later. For one input row,

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,\\
\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2.
\end{aligned}
$$

The subscript identifies the layer. The column $\boldsymbol\theta$ collects all
weights and biases mathematically; our code will store them in four tensors.
`X` has shape `(N, 2)` and contains only the original coordinates. Unlike
`X_affine`, it needs no extra column of ones: biases are stored separately.

`t.randn(rows, columns, requires_grad=True)` creates a matrix of random initial
values and asks PyTorch to track computations involving it. This will let us
differentiate a loss with respect to its entries later; it does not compute any
derivatives yet. With one size argument, `t.randn(size, requires_grad=True)`
creates a one-dimensional tensor. We use one-dimensional tensors for the biases:
adding a bias vector to a matrix adds it to each row (**broadcasting**).

Recall that `A.shape` gives a tensor's dimensions. You can count its scalar
entries from those dimensions, or use `A.numel()` to obtain that count directly.
Use separate `print(...)` statements if helpful; no printing loop is required.

**Exercise 1 — initialise the parameters.**

1. Work out the shapes of `W1`, `b1`, `W2` and `b2` from the matrix products above.
2. Create them using `t.randn(..., requires_grad=True)`. Use the dimension variables
   supplied below, and create them in the order **`b1`, `W1`, `b2`, `W2`** so that
   your random draws match the reference experiment.
3. Print their shapes. How many adjustable scalar parameters are there altogether?

The supplied `t.manual_seed(94)` fixes the sequence used to draw the
initial parameters. Running this cell again recreates the same starting values.
It does not change the already-created data `X` and labels `y`.

In [ ]:
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2; print shapes and count scalar parameters.

<details>
<summary>Show answer — parameter shapes and initialisation</summary>

The shapes are `W1`: `(2, 2)`, `b1`: `(2,)`, `W2`: `(2, 1)`, and `b2`: `(1,)`.
There are $2\times2+2+2\times1+1=9$ scalar parameters. The notes' bias rows are
represented by one-dimensional tensors here.

```python
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)
print('W1:', W1.shape, 'b1:', b1.shape)
print('W2:', W2.shape, 'b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

</details>

### 3.2. Implement the network calculation

The equations above describe one observation. Applying them to the whole matrix
`X` performs the same calculation for every row, using the same parameters.
`t.relu(A)` applies $\max\{0,a\}$ separately to every entry of `A`; it implements
the componentwise ReLU from the notes.

**Exercise 2 — write `net(X)`.** Translate the three equations into a Python
function returning the scores. You can use intermediate variables such as `a1`,
`phi1` and `scores`; these local names do not affect the supplied checks or plots.
Use the four tensors you have just created; do not draw new parameters inside
the function. Before running it, predict the shapes of all three intermediate
results for the dataset. What should happen if you pass only one input row as
`X[:1]`? This slice keeps a matrix with shape `(1, 2)`; `X[0]` would instead
return a one-dimensional tensor.

Use `@` for matrix multiplication and keep these calculations as PyTorch tensors.
The function must return its scores, not just print them. We will differentiate
this calculation later, so do not use `.item()` or `detach()` inside `net`.

The supplied checks after your implementation use `assert condition`: if the
condition is false, execution stops with an `AssertionError`. These checks test
the output shapes; they do not establish that every part of an implementation
is correct. If a check fails, inspect the intermediate shapes and formulas.

The hidden activations are values of learned feature functions. The final output
is a classification **score**, not a probability or a predicted label.

In [ ]:
# YOUR CODE: define net(X).

<details>
<summary>Show answer — the network calculation</summary>

`a1` and `phi1` have shape `(N, 2)`; `scores` has shape `(N, 1)`.
For a single-row input, `a1` and `phi1` have shape `(1, 2)`, and `scores` has
shape `(1, 1)`. The variables contain values for the inputs passed to the
function; the parameter tensors are reused.

```python
def net(X):
    a1 = X @ W1 + b1
    phi1 = t.relu(a1)
    scores = phi1 @ W2 + b2
    return scores
```

</details>

In [ ]:
# Supplied checks: one score per observation, also for a single-row input.
assert net(X).shape == y.shape
assert net(X[:1]).shape == (1, 1)
print('Dataset output shape:', net(X).shape)

### 3.3. Evaluate the empirical loss and mistakes

Recall the operations used to count mistakes in Lab 4:

- Comparisons such as `A > 0` and `A != B` act entry by entry and return tensors
  of `True`/`False` values. `!=` means "not equal".
- Summing a Boolean tensor with `.sum()` counts its `True` entries.
- `A.dtype` describes the type of its entries; `B.to(A.dtype)` converts `B` to
  that type. This lets Boolean predictions be represented numerically as 0 and 1.
- `.item()` extracts a Python number from a one-value tensor, and `int(...)`
  converts a number to a Python integer.

Keep predicted labels and observed labels at the same shape `(N, 1)` when you
compare them. Work out which comparison identifies an incorrect prediction.

**Exercise 3 — evaluate your initial model.**

1. Calculate its scores and empirical logistic loss using `loss_from_scores`.
2. Write `mistakes_from_scores(scores)`, returning the number of incorrect labels
   as a Python integer. Like the loss helper, it should use the fixed labels `y`.
   Reuse Lab 4's decision rule: class 1 for a strictly positive score and class 0
   otherwise, including when the score is zero.
3. Print the loss and the number of mistakes before fitting.

Do not add a sigmoid to the network output for the loss.
To repeat this initial evaluation after fitting, rerun Exercise 1 first.

In [ ]:
# YOUR CODE: define mistakes_from_scores, then evaluate the initial model.

<details>
<summary>Show answer — initial loss and mistakes</summary>

Compare each predicted label with its observed label and count disagreements.
The supplied initialisation gives loss about $130.990$ and 50 mistakes. These are
the randomly initialised model's results before any fitting updates.

```python
def mistakes_from_scores(scores):
    predicted_labels = (scores > 0).to(y.dtype)
    return int((predicted_labels != y).sum().item())

scores = net(X)
L = loss_from_scores(scores)
print('Initial empirical loss:', L.item())
print('Initial mistakes:', mistakes_from_scores(scores))
```

</details>

In [ ]:
# Supplied checks on artificial scores, including ties at zero.
assert mistakes_from_scores(t.zeros_like(y)) == 50
assert mistakes_from_scores(2 * y - 1) == 0
print('Mistake-count checks passed.')

### 3.4. Ask PyTorch for the derivatives

Here is a small example separate from the network. Predict the two partial
derivatives of $Q(u)=u_0^2+u_1^2$ at $u=(1,-2)$, then run the cell.
`Q.backward()` computes the derivatives and stores them in `u.grad`.

In [ ]:
u = t.tensor([1.0, -2.0], requires_grad=True)
Q = (u ** 2).sum()
Q.backward()
print('u:', u)
print('Gradient of Q:', u.grad)

The gradient is $(2,-4)$ and `u` has not changed. **`L.backward()` computes
derivatives of the scalar loss; it does not update the parameters.**

For our network, PyTorch saves these derivatives in `W1.grad`, `b1.grad`,
`W2.grad` and `b2.grad`. Each gradient has the same shape as its parameter:
`W1.grad[0, 0]` is the derivative with respect to `W1[0, 0]`. These parameter
tensors were created directly with `requires_grad=True` (they are *leaf tensors*).
Intermediate activations and scores do not normally retain their own `.grad`.

**Gradients accumulate.** If derivatives are already stored, `backward()` adds
to them. Computing a new loss does not clear them. Accumulation lets us add the
gradients of losses calculated separately; here our empirical loss already sums
over all observations, so we need just one derivative calculation per update.

Before each derivative calculation, we therefore set each parameter's `.grad`
to `None`, Python's value for “nothing stored”. This removes old gradients without
changing the parameters; the next `backward()` stores fresh gradients. The loop
below does this once for each of the four parameter tensors. It also works when
no gradients have been calculated yet.

PyTorch uses 0 as the ReLU multiplier at input 0. This is a computational
convention; the ordinary derivative remains undefined there.

**Exercise 4 — compute the network's gradients.** After the supplied clearing
code, calculate a fresh empirical loss, call `L.backward()`, and print each
parameter's shape and its gradient's shape. For example, compare `W1.shape`
and `W1.grad.shape`.

Explain why the shapes match. If one entry of `W1.grad` is positive, should the
corresponding weight increase or decrease in gradient descent with a positive
learning rate? These derivatives refer to the current parameters; rerun
Exercise 1 first if you want to examine the initial parameters again.

In [ ]:
# Supplied: clear previously stored gradients.
for parameter in (b1, W1, b2, W2):
    parameter.grad = None

# YOUR CODE: calculate the loss, differentiate it, and print the shapes.

<details>
<summary>Show answer — derivatives of the network loss</summary>

Each derivative corresponds to one parameter entry, so the shapes agree.
A positive derivative gives a negative change in that weight: gradient descent
subtracts the learning rate times the derivative.

```python
for parameter in (b1, W1, b2, W2):
    parameter.grad = None

L = loss_from_scores(net(X))
L.backward()
print('W1:', W1.shape, W1.grad.shape)
print('b1:', b1.shape, b1.grad.shape)
print('W2:', W2.shape, W2.grad.shape)
print('b2:', b2.shape, b2.grad.shape)
```

</details>

### 3.5. Implement gradient descent

You already know the update rule: subtract the learning rate times each current
partial derivative from the corresponding parameter.

**Using `with t.no_grad():` is the standard way to perform these manual
gradient-descent updates in PyTorch.** PyTorch has an internal **gradient-tracking
switch**. When it is on, operations involving our parameters with
`requires_grad=True` are recorded so that `backward()` can calculate derivatives.
When it is off, numerical calculations still run, but their operations are not
recorded for differentiation.

Here `t` is our name for PyTorch, and `t.no_grad()` creates an object that manages
this switch. **`with` is a Python keyword** that uses this object to:

1. Save the current setting and switch tracking off on entering the block.
2. Execute the indented lines with tracking off.
3. Restore the previous setting on leaving the block, even if an error occurs.

First calculate the loss and call `backward()` **outside** this block. Inside,
use the stored gradients to update the parameters. Only the tracking setting
is temporary: the parameter updates remain after the block. The gradients
already stored in `.grad` remain available, and the parameters still have
`requires_grad=True`. In our loop, tracking is back on for the next loss
calculation using the updated parameters.

Without this block, our first in-place parameter update with `.add_(...)` would
raise an error and stop the cell before changing that parameter. Calling
`backward()` does not itself switch tracking off.

**`.add_(...)` changes a tensor in place.** The following cell applies one update
to the separate toy variable `u` above. Then `u.grad = None` clears the stored
derivatives without changing `u`. Run the toy derivative cell again first if
you want to repeat this example from its initial values.

For our parameter tensors, use these in-place updates rather than assigning a
new tensor to each parameter name. This keeps the same parameter tensors, still
requiring gradients, available for the next derivative calculation.

In [ ]:
with t.no_grad():
    u.add_(-0.1 * u.grad)
u.grad = None
print('After the toy update:', u)  # (0.8, -1.6)
print('Cleared gradient:', u.grad)  # None

**Exercise 5 — adapt gradient descent to all four parameter tensors.**

Use 1,000 full-dataset updates with learning rate `0.001`. We keep these settings
for every neural network. The affine model uses a different learning rate because
its parameterisation differs; we compare the resulting fits, not iteration speed.

**To start again, rerun Exercise 1 before this cell.** Running only this cell
continues fitting the current parameters.

1. Create `loss_history` and `error_history` as new lists containing the initial
   loss and mistake count. Use `.item()` to store a loss as a Python number;
   `mistakes_from_scores` already returns an integer.
2. At the start of each iteration, set all four `.grad` attributes to `None`,
   as in Exercise 4. Compute a fresh loss and call `backward()` once.
3. Inside the supplied `no_grad` block, update all four parameter tensors.
   Replace `pass` (a placeholder that does nothing) with your updates.
4. Recompute the scores after updating and append the new loss and mistake count
   to the histories. All derivatives must be computed before any parameter changes.

Entry `step` must describe `step` completed updates: both histories will have
1,001 entries. Use these names for the supplied plots. If your lists have other
names, assign them to `loss_history` and `error_history` at the end of the cell.

Keep the supplied recording lines. `save_boundary(net)` evaluates the current
network on the plotting grid and saves values for a picture; it does not change
the parameters or the empirical loss. `boundary_scores[step]` stores that picture
under the completed update number, just like the affine checkpoints.

The first supplied line also records the initial hidden activations of the fixed
input `X[:1]` for Exercise 8. `.detach()` keeps their values without gradient
tracking; this record is not used to calculate the fitting loss.

In [ ]:
# Supplied: record the starting feature values for Exercise 8.
features_before = t.relu(X[:1] @ W1 + b1).detach()
learning_rate = 0.001
num_updates = 1000
boundary_scores = {}

# YOUR CODE: create loss_history and error_history, including update 0.

for step in range(1, num_updates + 1):
    # YOUR CODE: clear gradients, calculate a fresh loss and differentiate it.

    with t.no_grad():
        # YOUR CODE: update every parameter using its gradient.
        pass

    # YOUR CODE: recompute scores and append loss and mistakes to the histories.

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, num_updates):
        boundary_scores[step] = save_boundary(net)

<details>
<summary>Show answer — the gradient-descent loop</summary>

The update rule is the same as in Lab 4. `backward()` now calculates the
derivatives for all four tensors. We clear gradients before calculating them,
update the parameters, then measure the new loss and mistakes.

```python
# Supplied: record the starting feature values for Exercise 8.
features_before = t.relu(X[:1] @ W1 + b1).detach()
learning_rate = 0.001
num_updates = 1000
boundary_scores = {}

scores = net(X)
loss_history = [loss_from_scores(scores).item()]
error_history = [mistakes_from_scores(scores)]

for step in range(1, num_updates + 1):
    for parameter in (b1, W1, b2, W2):
        parameter.grad = None
    L = loss_from_scores(net(X))
    L.backward()

    with t.no_grad():
        b1.add_(-learning_rate * b1.grad)
        W1.add_(-learning_rate * W1.grad)
        b2.add_(-learning_rate * b2.grad)
        W2.add_(-learning_rate * W2.grad)

    scores = net(X)
    loss_history.append(loss_from_scores(scores).item())
    error_history.append(mistakes_from_scores(scores))

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, num_updates):
        boundary_scores[step] = save_boundary(net)
```

</details>

In [ ]:
# Supplied: check recording and display progress.
assert len(loss_history) == len(error_history) == num_updates + 1
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, num_updates):
    print(step, loss_history[step], error_history[step])

Run this supplied cell after the original two-neuron fit. It keeps the results
under names beginning with `narrow_` for later comparisons. Subsequent fitting
cells create new lists and dictionaries, leaving these saved results available.
To reproduce this experiment later, rerun Exercise 1, Exercise 5 and this cell.

In [ ]:
# Supplied: keep the original two-neuron results for comparisons.
assert W1.shape == (2, 2), 'Run the original two-neuron initialisation and fit first.'
narrow_loss_history = loss_history
narrow_error_history = error_history
narrow_boundary_scores = boundary_scores
narrow_grid_scores = boundary_scores[num_updates]
narrow_features_before = features_before

## 4. Examine what your network learned

### 4.1. Inspect the loss and mistakes

**Exercise 6 — interpret the loss and mistake curves.** Run the complete supplied
cell below. It uses the original two-neuron results preserved by the preceding
supplied cell in `narrow_loss_history` and `narrow_error_history`:

- Left: empirical logistic loss against completed updates, for both models.
- Middle: affine loss divided by $\log 2$, alongside its mistake count.
- Right: the same two quantities for your neural network.

How much does the network's loss decrease? Does its mistake count decrease in
the same way? Compare the final fit with the affine model. We will discuss these
questions together with the boundaries and feature values under **Pause and
interpret: what did two hidden neurons add?** below.

The scaling is for display only: each mistake contributes at least $\log 2$ to
the summed logistic loss, so mistakes cannot exceed loss divided by $\log 2$.
**The fitting updates used the unscaled empirical loss.**

In [ ]:
# Supplied: compare the affine classifier with the original two-neuron fit.
plot_first_comparison(affine_loss_history, affine_error_history,
                      narrow_loss_history, narrow_error_history)

### 4.2. Inspect the decision boundaries

A neural network's zero-score set need not be a single straight line.
Our supplied plots use scores on a grid to draw this set. The black
line separates light blue class-0 regions from light orange class-1
regions. At exactly zero, our classifier selects class 0.
The grid is only for drawing: the losses and mistakes still use the
same 100 observations.

**Exercise 7 — interpret the decision boundaries.**

Run the complete supplied cell below. It shows the affine fit and the original
two-neuron network after 1, 100 and 1,000 updates. The supplied saving cell kept
those three stages in `narrow_boundary_scores`; `narrow_error_history` supplies
the mistake counts. These remain the original results when we fit other networks
later. The counts refer to the observed dataset; the grid only draws the picture.

How does the boundary change during fitting? Does its final shape separate the
four clouds successfully? We will return to these questions under **Pause and
interpret: what did two hidden neurons add?** below.
Why do we use saved scores for the earlier stages instead of evaluating the
final fitted network three times?

In [ ]:
# Supplied: the affine fit and three stages of the original two-neuron fit.
plot_boundaries(
    [affine_grid_scores, narrow_boundary_scores[1],
     narrow_boundary_scores[100], narrow_boundary_scores[1000]],
    ['Affine: 1000 updates', '2-neuron NN: 1 update',
     '2-neuron NN: 100 updates', '2-neuron NN: 1000 updates'],
    [affine_error_history[-1], narrow_error_history[1],
     narrow_error_history[100], narrow_error_history[1000]])

<details>
<summary>Show answer — why the earlier stages need saved scores</summary>

The parameters currently contain the final fit: calling `net` again would show
that final model every time. `narrow_boundary_scores` holds the saved values at
each selected update; `narrow_error_history[step]` holds its mistake count.

</details>

### 4.3. Check the feature values before and after fitting

**Exercise 8 — inspect the representation of one fixed input.** Work with the
original two-neuron fit, before starting the later experiments.

The input is `x = X[:1]`. Its initial hidden activations were recorded in
`narrow_features_before`. Use the hidden-layer formula to calculate its current
activations from `W1` and `b1`. Print both entries before and after fitting,
and the largest absolute change. `t.abs(A)` takes absolute values entry by entry;
`A.max().item()` gives the largest entry as a Python number.

Did the input change? Which parameters determine these feature values? Would
adjusting only `W2` and `b2` change them? To repeat this exercise after a later
experiment, first reproduce the original fit as described above.

In [ ]:
# YOUR CODE: compare the same input's hidden activations before and after fitting.

<details>
<summary>Show answer — the same input, adjustable features</summary>

The input stays fixed. The hidden parameters `W1` and `b1` determine its feature
values; the output parameters only combine them into a score. Changing hidden
parameters can change the representation, but a given feature need not change
at every input. Compare the printed values in this run.

```python
x = X[:1]
features_after = t.relu(x @ W1 + b1)
print('Before:', narrow_features_before[0, 0].item(), narrow_features_before[0, 1].item())
print('After: ', features_after[0, 0].item(), features_after[0, 1].item())
print('Largest absolute change:', t.abs(features_after - narrow_features_before).max().item())
```

</details>

### Pause and interpret: what did two hidden neurons add?

1. Compare the empirical loss, mistake count and boundary with the affine model.
2. Which parameters define the hidden features, and which combine them into a score?
   Did we prescribe the lecture's XOR features ourselves?
3. Does reducing the empirical loss necessarily reduce the number of mistakes?
   Use these curves and the affine experiment to support your answer.

Write down your answers before opening the explanation.

<details>
<summary>Show answer — interpreting the two-neuron fit</summary>

**This run does not improve on the affine fit.** After 1,000 updates, the
two-neuron network has loss about 69.308 and 47 mistakes;
the affine model has loss 69.308 and 47 mistakes. The network can represent
more flexible boundaries, but this initialisation has not led to a good fit.

`W1` and `b1` define the hidden features; `W2` and `b2` combine them into scores.
We supplied an architecture and random initial values, not the lecture's features.

Loss and mistakes measure different things. Loss uses the scores continuously;
the predicted label changes only when a score crosses zero. In the affine run,
the loss decreased while mistakes rose from 33 to 47. A flat mistake count is
therefore not enough to conclude that optimisation has stopped.

These values use the stated seed and settings. Small numerical differences
across PyTorch versions or devices are possible.
</details>

## 5. Why did two features solve XOR in the lecture?

In the lectures, we constructed a network with just two hidden neurons whose
decision boundary correctly classified all four XOR inputs. Here the observations
are noisy, but is noise enough to explain such a poor fit? Or has gradient descent
failed to find suitable parameters, perhaps ending up near a suboptimal local minimum?

We first apply the lecture's constructed score to our noisy observations to see how
it compares with the fitted network. We then investigate whether two strategies
discussed in the lectures can help: **starting from a different initialisation**
and **changing the architecture**. We will try a wider hidden layer, then a network
with two hidden layers, and compare their empirical losses, mistake counts and
decision boundaries.

For the first comparison, use the same score we constructed in the lecture:

$$s(\boldsymbol x)=\operatorname{ReLU}(x_1-x_2)
+\operatorname{ReLU}(x_2-x_1)-\frac12=|x_1-x_2|-\frac12.$$

It is positive at the class-1 centres $(0,1)$ and $(1,0)$, and negative at the
class-0 centres $(0,0)$ and $(1,1)$.

**Exercise 9 — compare the ideal inputs with the noisy observations.**

1. Write `lecture_score(inputs)` for this score, returning one column of scores.
   Use slices such as `inputs[:, 0:1]` to preserve the column dimension.
2. Count its mistakes on the four supplied centres, using `centre_y`. Our helper
   `mistakes_from_scores` uses the 100 labels in `y`, so do not use it for the centres.
3. Evaluate the same score on the observed data `X`; now use `mistakes_from_scores`
   and `loss_from_scores` to report its mistakes and empirical loss.
4. Explain why the lecture's exact classification of four inputs no longer
   guarantees correct labels for every observation in the clouds. Do your results
   show that this finite sample cannot be classified perfectly by any network?
5. Compare this empirical loss with `narrow_loss_history[-1]`, the loss of your
   fitted two-neuron network. Both scores use the same architecture: can the
   parameters found by gradient descent be a global minimiser of its empirical loss?

Use this separate function without changing `net` or its learned parameters.

In [ ]:
centre_X = t.tensor([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

# YOUR CODE: define lecture_score and compare the centres with the noisy data.

<details>
<summary>Show answer — ideal XOR versus noisy observations</summary>

The constructed score makes **0 mistakes on the centres**, but **8 on the noisy
observations**, with empirical loss about $54.071$. Each observation keeps its
cloud's label while its coordinates are perturbed. Some observations move across
the boundary that correctly separates the four centres. The construction therefore
does not inherit a guarantee of zero mistakes on these new inputs.

This shows the effect of the noisy coordinates on this particular classifier.
It does **not** prove that no classifier can fit all 100 observed labels. Nor can
we attribute every error of our fitted network solely to noise: its result also
depends on the available functions, initialisation and optimisation.

The constructed score uses two hidden neurons, just like our fitted network,
but improves both its empirical loss ($54.071$ instead of $69.308$) and its
mistake count (8 instead of 47). We therefore know that the fitted parameters
are not a global minimiser of the empirical loss within this architecture.
This run illustrates an optimisation difficulty as well as noisy data; it does
not by itself establish that gradient descent has reached a local minimum.

```python
centre_X = t.tensor([[0.0, 0.0], [1.0, 1.0], [0.0, 1.0], [1.0, 0.0]])
centre_y = t.tensor([[0.0], [0.0], [1.0], [1.0]])

def lecture_score(inputs):
    difference = inputs[:, 0:1] - inputs[:, 1:2]
    return t.relu(difference) + t.relu(-difference) - 0.5

centre_scores = lecture_score(centre_X)
centre_mistakes = int(((centre_scores > 0).to(centre_y.dtype) != centre_y).sum().item())
noisy_scores = lecture_score(X)
print('Mistakes on the four centres:', centre_mistakes)
print('Mistakes on the noisy observations:', mistakes_from_scores(noisy_scores))
print('Empirical loss on the noisy observations:', loss_from_scores(noisy_scores).item())
```

</details>

### 5.1. Can a different initialisation improve the fit?

We test the lecture's heuristic of **trying different initialisations**, keeping
the architecture, data, empirical loss, learning rate and update count fixed.
Predict whether changing only the initial parameters could change the fitted boundary.

**Exercise 10 — fit the two-neuron network from a different starting point.**

**Part A: initialise again.** Reuse Exercise 1's four `t.randn(...)` lines in
the cell below, with the same shapes and draw order. The supplied seed is now
9 instead of 94. It selects a different repeatable set of starting parameters;
it does not change the existing `X` or `y`. Keep the original exercise unchanged.

In [ ]:
t.manual_seed(9)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2; print shapes and count scalar parameters.

<details>
<summary>Show answer — a new starting point</summary>

The shapes and nine-parameter architecture are unchanged. Only the seed changes.

```python
t.manual_seed(9)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 2
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)
print('W1:', W1.shape, 'b1:', b1.shape)
print('W2:', W2.shape, 'b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

</details>

**Part B: reuse your fitting loop.** Copy your completed Exercise 5 into the cell
below and run it after Part A, with the same learning rate and update count.
`net`, the loss and mistake-count functions all work with the new parameters.
The mathematics of the update is unchanged.

To repeat from this starting point, rerun Part A before Part B. Rerunning Part B
alone continues fitting the current parameters. Then run the supplied saving cell;
it keeps this experiment separately under names beginning with `restart_`.

In [ ]:
# YOUR CODE: copy your completed fitting cell from Exercise 5, after running Part A.

<details>
<summary>Show answer — reusing your gradient-descent loop</summary>

Use the complete fitting code from **Exercise 5** unchanged, after running
Exercise 10 Part A. The different seed in Part A is the only experimental change.
The loss, gradients, parameter updates and recording operations stay the same.

</details>

In [ ]:
# Supplied: keep the second two-neuron run separately.
assert W1.shape == (2, 2)
assert len(loss_history) == len(error_history) == 1001
restart_loss_history = loss_history
restart_error_history = error_history
restart_grid_scores = boundary_scores[num_updates]
print('Original: loss', narrow_loss_history[-1], 'mistakes', narrow_error_history[-1])
print('New start: loss', restart_loss_history[-1], 'mistakes', restart_error_history[-1])

**Part C: compare and interpret.** Run the supplied plots of the two saved runs.
The supplied functions draw the saved results from both runs.

1. Compare the final empirical losses, mistake counts and boundaries. Which run
   fits these observations better? What is the only experimental choice we changed?
2. How does this illustrate the lecture's strategy of trying different starting
   points? Would rerunning Part B alone explore a new starting point?
3. Do different results after 1,000 updates prove that the runs have converged
   to different local minima? Does the improved run establish a global minimum?

Write down your answers before opening the explanation.

In [ ]:
# Supplied: compare the two initialisations, keeping the architecture fixed.
plot_network_curves(
    ['Original initialisation', 'New initialisation'],
    [narrow_loss_history, restart_loss_history],
    [narrow_error_history, restart_error_history])

In [ ]:
# Supplied: final boundaries from the same architecture with two starting points.
plot_boundaries(
    [narrow_grid_scores, restart_grid_scores],
    ['Original initialisation', 'New initialisation'],
    [narrow_error_history[-1], restart_error_history[-1]])

<details>
<summary>Show answer — comparing initialisations</summary>

**The new starting point improves the fit in this experiment.** The first run
ends at loss 69.308 with 47 mistakes; the second reaches
33.462 with 9 mistakes. Both have nine parameters and
use the same data, empirical loss, learning rate and number of updates.

This illustrates the lecture's heuristic of trying different initialisations.
The new seed in Part A chooses a different starting point. Rerunning Part A and
Part B repeats that experiment; running Part B alone continues the current fit.
A different initialisation need not improve the result.

Different results after 1,000 updates do not prove convergence to different local
minima. Nor does this improvement establish a global minimum or better predictions
on unseen data. The constructed lecture score already showed that the original
fit was not globally optimal within this architecture.
</details>

## 6. What changes with 100 hidden neurons?

We now test **changing the architecture**. Increase the hidden width from
$d_1=2$ to $d_1=100$, keeping the same 100 observations and one scalar output.
The figures compare this fit with the original two-neuron run; the final summary
also includes the improved initialisation.

**Exercise 11 — widen and fit your network.**

**Part A.** Copy your initialisation from Exercise 1 into the new cell below and
set `hidden_features = 100`. Use the supplied seed 94.
Predict the shapes and total parameter count first. Which tensor keeps its shape?
Changing the dimensions also changes the initial parameter arrays.
Keep the earlier exercise cells unchanged.

In [ ]:
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 100
output_features = 1

# YOUR CODE: create b1, W1, b2 and W2; print shapes and count scalar parameters.

<details>
<summary>Show answer — initialising the wider network</summary>

The shapes are `W1`: `(2, 100)`, `b1`: `(100,)`, `W2`: `(100, 1)`, and `b2`: `(1,)`.
There are 401 scalar parameters. Only `b2` keeps its shape; the network formula
still applies with these dimensions.

```python
t.manual_seed(94)  # Supplied: repeatable starting parameters.
input_features = 2
hidden_features = 100
output_features = 1

b1 = t.randn(hidden_features, requires_grad=True)
W1 = t.randn(input_features, hidden_features, requires_grad=True)
b2 = t.randn(output_features, requires_grad=True)
W2 = t.randn(hidden_features, output_features, requires_grad=True)
print('W1:', W1.shape, 'b1:', b1.shape)
print('W2:', W2.shape, 'b2:', b2.shape)
print('Scalar parameters:', W1.numel() + b1.numel() + W2.numel() + b2.numel())
```

</details>

**Part B.** After Part A, copy your completed fitting cell from Exercise 5 into
the cell below and run it. Keep learning rate `0.001` and 1,000 updates.
Explain why the same network function and update code work with wider tensors.
To repeat from the initial parameters, rerun Part A before fitting again.

In [ ]:
# YOUR CODE: reuse your completed fitting cell from Exercise 5, after running Part A.

<details>
<summary>Show answer — the same update rule with more neurons</summary>

Use the complete fitting code from **Exercise 5** unchanged, after the 100-neuron
initialisation. The matrix and gradient shapes follow the parameter shapes.
The learning rate, update count and mathematical update rule stay the same.

</details>

Run the supplied cell to keep this experiment's results under names beginning
with `wide_`, then compare the fits using the supplied figures.

In [ ]:
# Supplied: keep the 100-neuron results for comparison.
assert W1.shape == (2, 100)
assert len(loss_history) == len(error_history) == 1001
wide_loss_history = loss_history
wide_error_history = error_history
wide_grid_scores = boundary_scores[num_updates]

**Exercise 12 — compare the two widths.**

**Part A: curves.** Run the complete supplied cell to compare the original
two-neuron fit with the 100-neuron fit. It uses the histories saved by the
preceding supplied cells under names beginning with `narrow_` and `wide_`.
Compare both the empirical losses and the mistake counts.

In [ ]:
# Supplied: compare the two widths from their saved histories.
plot_network_curves(
    ['2 neurons (original)', '100 hidden neurons'],
    [narrow_loss_history, wide_loss_history],
    [narrow_error_history, wide_error_history])

**Part B: boundaries.** Run the complete supplied cell to compare the affine
model and both network widths. Each panel uses the grid scores saved for that
experiment, so you do not need to rerun an earlier fit to draw its boundary.

Which model has the smaller empirical loss and fewer mistakes? How does the
wider network's boundary differ? Do these results guarantee a better fit for
every initialisation, or better predictions on unseen observations?

In [ ]:
# Supplied: compare the final affine and neural-network boundaries.
plot_boundaries(
    [affine_grid_scores, narrow_grid_scores, wide_grid_scores],
    ['Affine', '2 neurons (original)', '100 hidden neurons'],
    [affine_error_history[-1], narrow_error_history[-1], wide_error_history[-1]])

<details>
<summary>Show answer — comparing widths</summary>

The original two-neuron fit has loss 69.308 and 47 mistakes;
the 100-neuron network reaches 7.159 with 2 mistakes and
separates most observations in this run. Wider tensors also have different
initial parameters. This experiment does not guarantee better fitting for every
initialisation, a global minimum, or better predictions on unseen data.
The previous exercise already improved the two-neuron fit by changing only its seed.
</details>

## 7. What changes with two hidden layers?

We now try another architecture change. Relative to the **original two-neuron
network**, we increase the first hidden width from 2 to **5** and add a second
hidden layer with **7 neurons**, as in the lecture's diagram. This changes both
width and depth. We keep one scalar output because our
task is binary classification. Thus $L=3$ (including the output layer), with
$d_0=2$, $d_1=5$, $d_2=7$ and $d_3=1$.

The second hidden layer takes the first layer's activations as its inputs:

$$
\begin{aligned}
\boldsymbol a_1(\boldsymbol x)&=\boldsymbol x\boldsymbol W_1+\boldsymbol b_1,
&\boldsymbol\phi_1(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_1(\boldsymbol x)),\\
\boldsymbol a_2(\boldsymbol x)&=\boldsymbol\phi_1(\boldsymbol x)\boldsymbol W_2+\boldsymbol b_2,
&\boldsymbol\phi_2(\boldsymbol x)&=\operatorname{\mathbf{ReLU}}(\boldsymbol a_2(\boldsymbol x)),\\
f_{\boldsymbol\theta}(\boldsymbol x)&=\boldsymbol\phi_2(\boldsymbol x)\boldsymbol W_3+\boldsymbol b_3.
\end{aligned}
$$

Here $W_2,b_2$ belong to **hidden layer 2**; $W_3,b_3$ form the output layer.
The output layer still forms an affine combination of the hidden activations,
with no ReLU or sigmoid applied to the resulting score.
Both hidden layers' parameters will be fitted: the second layer combines learned
features to produce another learned representation of the original input.

In Python, prefix this experiment's names with `deep_`, for example `deep_W1` and
`deep_b1`. This keeps the earlier `net`, parameters and saved results available.
The prefix only identifies the experiment; the numbers still identify layers.
Use `deep_net(X)` for the new network. The loss function, mistake-count function,
dataset and plotting helper can all be reused without changes.

### 7.1. Construct the new architecture

**Exercise 13 — initialise all six parameter tensors.**

1. Use the matrix products above to derive the shapes of `deep_W1`, `deep_b1`,
   `deep_W2`, `deep_b2`, `deep_W3` and `deep_b3`. Keep biases one-dimensional, as
   before, so they are added to every observation's row.
2. Use `t.randn(..., requires_grad=True)` and the supplied dimension variables.
   Draw in the order **`deep_b1`, `deep_W1`, `deep_b2`, `deep_W2`, `deep_b3`,
   `deep_W3`** to reproduce the reference experiment.
3. Print all six shapes and calculate the total number of scalar parameters,
   using their dimensions or `.numel()`.

The supplied seed 94 makes this initialisation repeatable. Keep the previous
experiments unchanged and work in these new cells.

In [ ]:
t.manual_seed(94)  # Supplied: repeatable starting parameters.
deep_input_features = 2
deep_hidden1_features = 5
deep_hidden2_features = 7
deep_output_features = 1

# YOUR CODE: create the six tensors in the stated order; print shapes and count entries.

<details>
<summary>Show answer — parameter tensors for two hidden layers</summary>

The shapes are `deep_W1`: `(2, 5)`, `deep_b1`: `(5,)`, `deep_W2`: `(5, 7)`,
`deep_b2`: `(7,)`, `deep_W3`: `(7, 1)` and `deep_b3`: `(1,)`.
There are $(2\times5+5)+(5\times7+7)+(7\times1+1)=65$ adjustable scalars.
The second matrix maps five hidden activations to seven preactivations; the last
matrix maps seven activations to one output score.

```python
t.manual_seed(94)  # Supplied: repeatable starting parameters.
deep_input_features = 2
deep_hidden1_features = 5
deep_hidden2_features = 7
deep_output_features = 1

deep_b1 = t.randn(deep_hidden1_features, requires_grad=True)
deep_W1 = t.randn(deep_input_features, deep_hidden1_features, requires_grad=True)
deep_b2 = t.randn(deep_hidden2_features, requires_grad=True)
deep_W2 = t.randn(deep_hidden1_features, deep_hidden2_features, requires_grad=True)
deep_b3 = t.randn(deep_output_features, requires_grad=True)
deep_W3 = t.randn(deep_hidden2_features, deep_output_features, requires_grad=True)

print('deep_W1:', deep_W1.shape)
print('deep_b1:', deep_b1.shape)
print('deep_W2:', deep_W2.shape)
print('deep_b2:', deep_b2.shape)
print('deep_W3:', deep_W3.shape)
print('deep_b3:', deep_b3.shape)
deep_parameter_count = (deep_W1.numel() + deep_b1.numel()
                        + deep_W2.numel() + deep_b2.numel()
                        + deep_W3.numel() + deep_b3.numel())
print('Scalar parameters:', deep_parameter_count)

```

</details>

**Exercise 14 — implement `deep_net(X)`.**

Translate the five equations into a function using the six `deep_` parameter
tensors. You can use intermediate variables such as `a1`, `phi1`, `a2`, `phi2`
and `scores`; their local names are up to you.
The input to the second hidden layer must be the first hidden layer's activations.
Return the final scores; do not draw parameters, detach tensors or convert them to
Python numbers inside the function.

Before running it, predict the shapes of all five intermediate tensors for the
full dataset and for the single-row slice `X[:1]`. How many feature values
describe each observation after the first hidden layer? After the second?

Use the existing `loss_from_scores` and `mistakes_from_scores` to report the
initial empirical loss and mistakes.
Explain why neither helper needs to know how many hidden layers there are.

In [ ]:
# YOUR CODE: define deep_net(X), then evaluate its initial loss and mistakes.

<details>
<summary>Show answer — composing two hidden layers</summary>

For the full dataset, `a1` and `phi1` have shape `(100, 5)`, `a2` and `phi2`
have shape `(100, 7)`, and `scores` has shape `(100, 1)`. With one input row,
replace the first dimension by 1. Each observation has five first-layer feature
values and seven second-layer feature values. All observations share the same
weights and biases.

The two helpers only need the scalar score for each observation and the labels
in `y`. They evaluate the same empirical loss and classification rule regardless
of how the network produced those scores. With the stated initialisation, the
initial loss is about $152.704$, with 60 mistakes.

```python
def deep_net(X):
    a1 = X @ deep_W1 + deep_b1
    phi1 = t.relu(a1)
    a2 = phi1 @ deep_W2 + deep_b2
    phi2 = t.relu(a2)
    scores = phi2 @ deep_W3 + deep_b3
    return scores

deep_initial_scores = deep_net(X)
print('Initial empirical loss:', loss_from_scores(deep_initial_scores).item())
print('Initial mistakes:', mistakes_from_scores(deep_initial_scores))
```

</details>

In [ ]:
# Supplied checks: the output remains one scalar score per observation.
assert deep_net(X).shape == y.shape
assert deep_net(X[:1]).shape == (1, 1)

### 7.2. Fit every layer

**Exercise 15 — adapt your gradient-descent loop to six tensors.**

Use Exercise 5 as your starting point, with learning rate `0.001` and 1,000 updates.
To repeat this experiment from the beginning, rerun Exercise 13 before fitting.

1. Create new lists `deep_loss_history` and `deep_error_history`, including update 0.
2. At each iteration, clear all six gradients with `parameter.grad = None`, then
   calculate the loss from `deep_net(X)` and call `backward()` once.
3. Inside `no_grad`, update all six tensors. After leaving it, recompute the scores
   and append the updated loss and mistake count to the histories.

Keep the supplied snapshot lines and these history names so the plots can find
the results. Both histories should have 1,001 entries. If you used other names,
assign your lists to these two names at the end, as in Exercise 5.

Which parts of the loop change? Which mathematical operations stay the same?
What would happen if you forgot to update the output layer, or calculated the
loss for `backward()` inside `no_grad`?

In [ ]:
deep_learning_rate = 0.001
deep_num_updates = 1000
deep_boundary_scores = {}

# YOUR CODE: create deep_loss_history and deep_error_history, including update 0.

for step in range(1, deep_num_updates + 1):
    # YOUR CODE: clear gradients, calculate a fresh loss and differentiate it.

    with t.no_grad():
        # YOUR CODE: update every parameter using its gradient.
        pass

    # YOUR CODE: recompute scores and append loss and mistakes to the histories.

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, deep_num_updates):
        deep_boundary_scores[step] = save_boundary(deep_net)

<details>
<summary>Show answer — fitting both hidden layers and the output</summary>

The model and parameter names change, but the loss, differentiation and update
rule are the same. All six gradients are evaluated before any parameter changes.
Omitting an output update would leave that parameter fixed. Computing the loss
inside `no_grad` would prevent the derivative calculation from being recorded.

```python
deep_learning_rate = 0.001
deep_num_updates = 1000
deep_boundary_scores = {}

scores = deep_net(X)
deep_loss_history = [loss_from_scores(scores).item()]
deep_error_history = [mistakes_from_scores(scores)]

for step in range(1, deep_num_updates + 1):
    for parameter in (deep_b1, deep_W1, deep_b2, deep_W2, deep_b3, deep_W3):
        parameter.grad = None
    L = loss_from_scores(deep_net(X))
    L.backward()

    with t.no_grad():
        deep_b1.add_(-deep_learning_rate * deep_b1.grad)
        deep_W1.add_(-deep_learning_rate * deep_W1.grad)
        deep_b2.add_(-deep_learning_rate * deep_b2.grad)
        deep_W2.add_(-deep_learning_rate * deep_W2.grad)
        deep_b3.add_(-deep_learning_rate * deep_b3.grad)
        deep_W3.add_(-deep_learning_rate * deep_W3.grad)

    scores = deep_net(X)
    deep_loss_history.append(loss_from_scores(scores).item())
    deep_error_history.append(mistakes_from_scores(scores))

    # Supplied: record the boundary at these completed updates.
    if step in (1, 100, deep_num_updates):
        deep_boundary_scores[step] = save_boundary(deep_net)
```

</details>

In [ ]:
# Supplied: check recording and display progress.
assert len(deep_loss_history) == len(deep_error_history) == deep_num_updates + 1
print('Updates   Empirical logistic loss   Mistakes')
for step in (0, 1, 10, 100, deep_num_updates):
    print(step, deep_loss_history[step], deep_error_history[step])

### 7.3. Compare width and depth

**Exercise 16 — compare all three neural networks.**

Run the three complete supplied plotting cells below, then answer the questions
under **Interpret the architecture and initialisation experiments**. The
two-neuron curves and boundaries use its original initialisation. The final
table includes the second initialisation as well, so keep that comparison in mind.

**Part A: loss and mistakes.** Compare the saved histories for all three
architectures. The deeper network's results are in `deep_loss_history` and
`deep_error_history`. Each history includes updates 0 through 1,000.

In [ ]:
# Supplied: compare all three architectures from their saved histories.
plot_network_curves(
    ['2 neurons (original)', '100 hidden neurons', '5 + 7 hidden neurons'],
    [narrow_loss_history, wide_loss_history, deep_loss_history],
    [narrow_error_history, wide_error_history, deep_error_history])

**Part B: final boundaries.** Run the supplied cell to compare all three
architectures after 1,000 updates. It uses the saved final grid scores for each
experiment. How does the deeper network's boundary differ from the other two?

In [ ]:
# Supplied: compare all three final decision boundaries.
plot_boundaries(
    [narrow_grid_scores, wide_grid_scores, deep_boundary_scores[1000]],
    ['2 neurons (original)', '100 hidden neurons', '5 + 7 hidden neurons'],
    [narrow_error_history[-1], wide_error_history[-1], deep_error_history[-1]])

**Part C: progress during fitting.** Run the supplied cell to see the deeper
network after 1, 100 and 1,000 updates. These panels use the grid snapshots in
`deep_boundary_scores`. Compare how the boundary and mistake count change.

In [ ]:
# Supplied: follow the deeper network through its saved stages.
plot_boundaries(
    [deep_boundary_scores[1], deep_boundary_scores[100], deep_boundary_scores[1000]],
    ['Deeper NN: 1 update', 'Deeper NN: 100 updates', 'Deeper NN: 1000 updates'],
    [deep_error_history[1], deep_error_history[100], deep_error_history[1000]])

### Interpret the architecture and initialisation experiments

The supplied table below collects the final measurements. The parameter counts
are those you derived in Exercises 1, 11 and 13. Use it together with your plots
to answer these questions before opening the interpretation:

1. Which runs illustrate trying another initialisation, and which illustrate
   changing the architecture? Relative to the original two-neuron network,
   which architecture change increases only width, and which also adds depth?
2. How much does the deeper network improve between its first and its 1,000th
   update? Compare this with the change from update 0 to update 1.
3. How does it compare with the two single-hidden-layer networks, including the
   number of parameters and the shapes of the boundaries? Include the two-neuron
   restart in your comparison using the table below.
4. Have we isolated the effect of depth, given that widths, parameter counts and
   initial parameter arrays also differ?
5. Does a smaller empirical loss establish a global minimum or better predictions
   on unseen data? Does a flat mistake curve mean fitting has stopped?

In [ ]:
# Supplied: collect your final measurements for interpretation.
print('Hidden neurons                 Parameters   Final loss   Mistakes')
print(f'2, original initialisation              9   {narrow_loss_history[-1]:10.3f}   {narrow_error_history[-1]}')
print(f'2, new initialisation                   9   {restart_loss_history[-1]:10.3f}   {restart_error_history[-1]}')
print(f'100                                   401   {wide_loss_history[-1]:10.3f}   {wide_error_history[-1]}')
print(f'5 then 7                               65   {deep_loss_history[-1]:10.3f}   {deep_error_history[-1]}')

<details>
<summary>Show answer — architecture and initialisation</summary>

The second two-neuron run changes only the initialisation. The 100-neuron network
increases width. Relative to the original network, the 5-then-7 network increases
the first width and adds a hidden layer. All four runs use the same data, loss,
learning rate and number of updates.

The deeper network has 42 mistakes after one update,
15 after 100, and 4 after 1,000. Its loss
falls from 152.704 to 136.520 after one update
and to 13.154 by the end. The later updates still matter.

Compare the final results: the original two-neuron network (9 parameters) has
loss 69.308 and 47 mistakes; its new initialisation gives
33.462 and 9. The 100-neuron network (401 parameters)
gives 7.159 and 2; the deeper network (65 parameters) gives
13.154 and 4. Both architecture changes and the new
initialisation improve on the original fit in these runs.

These are examples of the lecture's heuristic strategies. More parameters do not
guarantee a better result. The architecture comparison does not isolate depth:
widths, parameter counts and initial values also change. Neither strategy
guarantees a global minimum or better predictions on new observations.
A flat mistake count does not mean fitting has stopped: scores and empirical
loss can still change without any predicted label changing.
</details>

You have translated a network architecture into tensors, implemented its score
function, calculated empirical losses and mistakes, obtained derivatives, and
used them in your own gradient-descent loop. The supplied plots of your results
let you compare the fitted network with the affine model on the same data. You
also inspected how changing hidden parameters changes the feature representation,
compared ideal XOR inputs with noisy observations, tried another initialisation,
widened the network, and implemented a second hidden layer. Across the
neural-network experiments, we kept the dataset, empirical loss, learning rate
and update count fixed, while changing either the starting parameters or
the architecture.

You have therefore tested two of the lecture's heuristic strategies for finding
useful solutions: trying different initialisations and changing the architecture.
Their effect must be checked in the experiment rather than assumed.